---
title: "DRG Cleaning v2"

author: "Carlos Resurreccion"

date: "2024-10-21"

---


# !!! MAKE SURE YOU RESTART THE (JUPYTER) R KERNEL BEFORE PROCEEDING !!!


# Parameters

Change which year to process in
`~/drg-pipeline/data-cleaning/cache/year_to_load.txt`

Change main GLOBAL (i.e. across all scripts) parameters in 
`~/drg-pipeline/data-cleaning/00a-parameters.r`


Change seldom touched parameters in
`~/drg-pipeline/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R`

In [49]:
source("~/drg-pipeline/data-cleaning/00a-parameters.r")


Parallelization: TRUE 


# Libraries


In [50]:
# Update the grouper
system("git submodule update --init --recursive")

# List required packages
required_packages <- c(
  "data.table", # Fast data manipulation
  "here", # Simplifies file path management
  "tictoc", # Timing code execution
  "stringr", # String manipulation
  "stringi", # Unicode string processing
  "lubridate", # Date-time handling
  "profvis", # Profiling R code
  "hash", # Hashing utility
  "future", # Parallel processing
  "future.apply", # Parallelized apply functions
  "knitr", # Dynamic report generation
  "htmlwidgets", # Interactive HTML widgets
  "parallelly", # Advanced parallel computing
  "stringdist", # String distance calculations
  "parallel", # Base parallel computing
  "reticulate", # Interface to Python
  "bigrquery", # BigQuery client
  "jsonlite", # JSON parsing
  "googleCloudStorageR", # Google Cloud Storage access
  "haven", # Read/write Stata, SPSS, SAS files
  "fst", # Fast serialization
  "httr", # HTTP requests
  "ggplot2", # Data visualization
  "rmarkdown", # Dynamic markdown documents
  "digest", # Create cryptographic hashes
  "base64enc", # Base64 encoding/decoding
  "arrow", # Apache Arrow for fast data storage
  "tidyverse" # Collection of data science packages
)

github_packages <- c(
  "r-lib/styler" # Code formatting
)

# Installation commands (commented out, for reference)
# invisible(lapply(required_packages, function(pkg) if (!require(pkg, character.only = TRUE)) install.packages(pkg)))
# invisible(lapply(github_packages, function(repo) if (!require(basename(repo), character.only = TRUE)) remotes::install_github(repo)))

# Load packages (assumes they are already installed)
invisible(lapply(required_packages, library, character.only = TRUE))
invisible(lapply(basename(github_packages), library, character.only = TRUE))


# R Scripts


In [51]:
year_to_load <- 2018

# Source each file sequentially
for (file in list.files(here::here("data-cleaning/r_scripts_v2"), pattern = "\\.R$", full.names = TRUE)) invisible(source(file))

message(year_to_load)


All directories exist.


Total Rows via cached object: 11777674

Utilizing 4 cores (8 threads)


2018



# Data Cleaning Proper


# Load Mapping Data


In [ ]:
# Enable caching and printing options
to_use_cache <- TRUE # Use .rds cache files
to_print_mapping_data <- FALSE # Print mapping data tables

# Load data from cache or query from BigQuery
load_or_query <- function(query, var_name) {
  rds_path <- here(cache_path, "mapping", paste0(var_name, ".rds"))
  if (to_use_cache && file.exists(rds_path)) {
    if (verbose_output) message("Loading ", var_name, " from cache...")
    return(readRDS(rds_path))
  }
  if (verbose_output) message("Querying ", var_name, " from BigQuery...")
  dt <- query_bq_to_dt(query)
  saveRDS(dt, rds_path)
  return(dt)
}

# Load procedure codes
tdrg_proc <- load_or_query(paste0("SELECT * FROM ", gcp_proj, ".grouper_v5.proc"), "proc")
tdrg_proc[, CODE := as.character(CODE)]

# Load RVS to ICD-9 mapping
rvs_icd9 <- load_or_query(paste0("SELECT * FROM ", gcp_proj, ".phic_libraries.acr_rvs_map"), "rvs_icd9")
rvs_icd9 <- rvs_icd9[, .(rvs = as.character(rvs), icd9cm = as.character(as.numeric(icd9cm) * 100))]

# Merge with DRG procedure data
rvs_icd9 <- merge(rvs_icd9, tdrg_proc[, .(CODE, DRGUSE)], by.x = "icd9cm", by.y = "CODE", all.x = TRUE)
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), .(rvs, icd9cm, is_drg = !is.na(DRGUSE) & DRGUSE)]

# Load additional datasets
acr_rvs <- load_or_query(paste0("SELECT * FROM ", gcp_proj, ".phic_libraries.acr_procedure"), "acr_rvs")
tdrg_icd10 <- load_or_query(paste0("SELECT * FROM ", gcp_proj, ".grouper_v5.i10"), "tdrg_icd10")
setkey(tdrg_icd10, "CODE")

# Get accepted ICD-10 codes
acc_pdx <- unique(tdrg_icd10[ACCPDX == "Y", CODE])
acc_pdx_env <- new.env(hash = TRUE)
for (code in acc_pdx) assign(code, TRUE, envir = acc_pdx_env)

# Load Philippine-specific ICD-10 codes
phl_icd10 <- load_or_query(paste0("SELECT * FROM ", gcp_proj, ".icd.phl_icd10"), "phl_icd10")

# Extract neoplasm codes
neoplasms_dt_actual <- as.data.table(phl_icd10[grepl("/", icd10), .(icd10 = sapply(strsplit(icd10, ","), function(x) trimws(x[2])))])

# Load expanded ICD-10 validation dataset
i10vx <- load_or_query(paste0("SELECT * FROM ", gcp_proj, ".grouper_v5.i10vx"), "i10vx")
setkey(i10vx, "code")
acc_icd_set <- unique(i10vx$code)

# Load healthcare institution data
hci <- load_or_query(paste0("SELECT * FROM ", gcp_proj, ".hci.temp_hci"), "hci")

# Create lookup environments
create_env_from_vector <- function(vec) {
  env <- new.env(parent = emptyenv())
  list2env(setNames(as.list(rep(TRUE, length(vec))), vec), envir = env)
  return(env)
}

proc_env <- create_env_from_vector(tdrg_proc$CODE)
rvs_env <- create_env_from_vector(rvs_icd9$rvs)
icd9cm_env <- create_env_from_vector(rvs_icd9$icd9cm)
acr_rvs_env <- create_env_from_vector(acr_rvs$rvs)
acc_pdx_env <- create_env_from_vector(acc_pdx)
phl_icd10_env <- create_env_from_vector(phl_icd10$icd10)
acc_icd_env <- create_env_from_vector(acc_icd_set)
hci_env <- create_env_from_vector(hci$id_hci)

# Restore proper patterns
neoplasm_codes <- unique(neoplasms_dt_actual$icd10)
covid_codes <- unique(covid_rvs)
rvs_codes <- unique(acr_rvs$rvs)

covid_rvs_neoplasm_pattern <- paste0("(", paste(unique(c(covid_codes, rvs_codes, neoplasm_codes)), collapse = "|"), ")")


Part 2: Main Data Cleaning Loop


In [53]:
# Data Cleaning Pipeline for DRG Processing
# This script processes large datasets in parts, applying parallel processing for efficiency.
# It reads, chunks, processes, and consolidates data before saving intermediate and final outputs.

for (loop_part in 1:split_parts) {
  start_time <- Sys.time() # Record start time for processing

  # Step 1: Read the appropriate file
  cat(paste0("\rStart reading part ", loop_part, " of ", split_parts))
  flush.console()
  read_result <- read_appropriate_file(loop_part)
  read_in_dt <- read_result$read_result_dt

  cat(paste0("\rFinished reading part ", loop_part, " of ", split_parts))
  flush.console()

  # Step 2: Split the data into chunks for parallel processing
  cat(paste0("\rStart chunking part ", loop_part, " of ", split_parts))
  flush.console()
  chunk_size <- ceiling(nrow(read_in_dt) / nthreads)
  chunks <- split(
    read_in_dt,
    rep(1:nthreads, each = chunk_size, length.out = nrow(read_in_dt))
  )
  cat(paste0("\rFinished chunking part ", loop_part, " of ", split_parts))
  flush.console()

  # Step 3: Process chunks in parallel or sequentially
  cat(paste0("\rStart processing part ", loop_part, " of ", split_parts))
  flush.console()
  parallel_results <-
    if (to_parallel) {
      mclapply(chunks, process_chunk, mc.cores = nthreads)
    } else if (!to_debug) {
      lapply(chunks, process_chunk)
    } else {
      list(process_chunk(chunks[[1]]))
    }

  # Consolidate processed chunks
  summarized_dt <- rbindlist(parallel_results)

  # Step 4: Save processed data if required
  if (to_write) {
    saveRDS(
      summarized_dt, here(checkpoint_1_path, paste0(
        checkpoint_1_prefix, year_to_load, suffix,
        "part_", sprintf("%02d", loop_part), "_of_", split_parts, ".rds"
      )),
      compress = TRUE
    )
  }

  # Step 5: Log processing time and update status
  processing_times[[loop_part]] <- as.numeric(difftime(Sys.time(), start_time, units = "secs"))
  print_status_update(loop_part, split_parts, processing_times, "clean")

  # Cleanup memory
  rm(read_in_dt, summarized_dt)
  invisible(gc())
}

# Step 6: Combine all processed parts into a master data table
master_dt_list <- parallel::mclapply(1:split_parts, function(read_part) {
  cat(paste("\rStarted reading part", read_part))
  flush.console()
  return_dt <- readRDS(here(checkpoint_1_path, paste0(
    checkpoint_1_prefix, year_to_load, suffix,
    "part_", sprintf("%02d", read_part), "_of_", split_parts, ".rds"
  )))
  cat(paste("\rFinished reading part", read_part))
  flush.console()
  return(return_dt)
}, mc.cores = nthreads)

# Merge all parts into a single data table
message("Commencing rbindlist")
master_dt <- rbindlist(master_dt_list, fill = TRUE)
rm(master_dt_list)
invisible(gc())
message("Finished rbindlist")

# Step 7: Save final processed data
if (to_write) {
  message("Commencing saveRDS")
  saveRDS(master_dt, here(
    checkpoint_2_path, paste0(
      checkpoint_2_prefix, year_to_load, suffix, ".rds"
    )
  ), compress = TRUE)
  message("Finished saveRDS")
}

# Save a pre-final version of the master dataset
message(paste0("Saving ", paste0(checkpoint_2_prefix, year_to_load, suffix, "prefinal", ".rds")))
saveRDS(master_dt, here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "prefinal", ".rds")
), compress = TRUE)
message(paste0("Finished saving ", paste0(checkpoint_2_prefix, year_to_load, suffix, "prefinal", ".rds")))


Start processing part 1 of 155

Finished cleaning 15 of 15 parts in 33s (ETA 0s)        

Commencing rbindlist

Finished rbindlist

Commencing saveRDS

Finished saveRDS

Saving checkpoint_2_claims_2018_sampled_625_prefinal.rds

Finished saving checkpoint_2_claims_2018_sampled_625_prefinal.rds



# Temp Output for Verification of Refactor

In [54]:
data <- readRDS("/home/resurreccion_cmc/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_2_master_clean_claims/checkpoint_2_claims_2018_sampled_625_prefinal.rds")
fwrite(data, "test.csv")


# BQ Preparation

In [55]:
# Final preparations for BQ upload

# Load the dataset from the prefinal checkpoint
result <- readRDS(here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "prefinal", ".rds")
))

# Add is_covid variable
# Identifies COVID-related claims by checking multiple clinical fields
result[, is_covid := {
  covid_found <- rep(FALSE, .N) # Initialize all rows as FALSE

  # Check each field sequentially, marking matches as TRUE
  not_found <- !covid_found
  covid_found[not_found] <- clin_c1[not_found] %chin% covid_rvs # Check primary diagnosis

  not_found <- !covid_found
  covid_found[not_found] <- clin_c2[not_found] %chin% covid_rvs # Check secondary diagnosis

  not_found <- !covid_found
  covid_found[not_found] <- c2[not_found] %chin% covid_rvs # Check coded diagnosis

  not_found <- !covid_found
  covid_found[not_found] <- c1[not_found] %chin% covid_rvs # Check additional coded diagnosis

  not_found <- !covid_found
  covid_found[not_found] <- sapply(clin_rvs[not_found], function(row) any(row %chin% covid_rvs)) # Check procedure codes

  not_found <- !covid_found
  covid_found[not_found] <- sapply(clin_sdx[not_found], function(row) any(row %chin% covid_rvs)) # Check supporting diagnoses

  not_found <- !covid_found
  covid_found[not_found] <- sapply(clin_proc[not_found], function(row) any(row %chin% covid_rvs)) # Check performed procedures

  covid_found # Return logical vector of COVID matches
}]

# Subset the dataset for BQ
# Keep only relevant columns needed for BigQuery upload
result <- result[, .(
  id_series, id_pin, id_hci, id_hcp, # Identifiers
  date_adm, date_dis, date_rec, date_ref, date_check, # Date-related fields
  pat_type, pat_rel, pat_age, pat_ageday, pat_sex, pat_bwt, pat_memcat_parent, pat_memcat_child, # Patient details
  claim_status, claim_payout, claim_charge, is_covid, # Claim-related fields
  clin_discharge, clin_outpatient, clin_emergency, clin_acc, # Clinical classification
  clin_c1, clin_c2, clin_sdx, clin_proc, clin_pdx, clin_pdx_source # Clinical details
)]

# Save the processed dataset to a new checkpoint before BQ upload
saveRDS(result, here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")
))


# BQ Upload

In [56]:
# BQ upload
if (to_bq) {
  if (!to_sample) bq_table <- paste0("claims_", year_to_load) # Define BQ table name

  # Attempt to delete the table if it exists
  tryCatch(
    bq_table_delete(bq_table(gcp_proj, bq_dataset, bq_table)),
    error = function(e) {
      if (grepl("Not found", e, ignore.case = TRUE)) {
        message("Table does not exist, nothing to drop.")
      } else {
        stop(e)
      }
    }
  )

  # Create the BQ table if it does not exist
  tryCatch(
    bq_table_create(
      bq_table(gcp_proj, bq_dataset, bq_table),
      fields = fromJSON(here(
        "data-cleaning/r_scripts_v2",
        "bq_schema_cleaning.json"
      ), simplifyDataFrame = FALSE)
    ),
    error = function(e) {
      if (grepl("already exists", e, ignore.case = TRUE)) {
        message("Table already exists. Skipping creation and upload.")
      } else {
        stop(e)
      }
    }
  )

  if (to_write) {
    chunk_size <- 250000 # Define chunk size for upload
    num_chunks <- ceiling(nrow(result) / chunk_size) # Calculate number of chunks

    for (i in seq_len(num_chunks)) {
      chunk <- result[((i - 1) * chunk_size + 1):min(i * chunk_size, nrow(result)), ] # Extract chunk

      # Upload chunk to BQ
      bq_table_upload(
        bq_table(gcp_proj, bq_dataset, bq_table),
        values = chunk,
        write_disposition = if (i == 1) "WRITE_EMPTY" else "WRITE_APPEND"
      )
    }
  }
}
